### Notebook 3 — Jointures et agrégations

In [122]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, year, col, round, concat_ws
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.functions import trim, initcap, upper, col
from collections import Counter

##### SparkSession

In [123]:
spark = SparkSession.builder.appName("TradeCorp ETL").getOrCreate()

##### chargement de parquet files pour la transformation

In [124]:
#Lecture simple d’un Parquet

DATA_PATH = "/home/jovyan/data/tmp/clean"   # adapte selon ton volume Docker

# Lecture des fichiers Parquet
df_categories = spark.read.parquet(f"{DATA_PATH}/categories/")
df_customers = spark.read.parquet(f"{DATA_PATH}/customers_clean/")
df_employees = spark.read.parquet(f"{DATA_PATH}/employees_clean/")
df_supply = spark.read.parquet(f"{DATA_PATH}/en_stock/")
df_shippers = spark.read.parquet(f"{DATA_PATH}/is_shipped/")
df_order_details = spark.read.parquet(f"{DATA_PATH}/order_details_final/")
df_orders = spark.read.parquet(f"{DATA_PATH}/orders_1997/")
df_products = spark.read.parquet(f"{DATA_PATH}/products_filtered/")

# Dictionnaire des DataFrames
dfs = {
    "customers_clean": df_customers,
    "employees_clean": df_employees,
    "en_stock": df_supply,
    "is_shipped": df_ship,
    "order_details_final": df_order_details,
    "orders_1997": df_orders,
    "orders_customers": df_orders_customers,
    "products_filtered": df_products,    
}

# Vérification
#df_orders.show(5)
#df_orders.printSchema()

##### Écriture en Parquet (Silver layer)

##### A21 — Jointure orders + customers

In [125]:
# #Création de orders_customers silver parquet file
# volumes pour silver 
DATA_PATH_silver = "/home/jovyan/data/tmp/silver" 

df_orders_customers = (
    df_orders.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="inner")
    .select(
        "o.order_id",
        "c.company_name",
        "c.country",
        "o.order_date",
        "o.freight"
    )
)

#Vérification
df_orders_customers.show(10)
df_orders_customers.printSchema()

# # Écriture en Silver

df_order_details_products.write.mode("overwrite").parquet(f"{DATA_PATH_silver}/orders_customers")

+--------+--------------------+---------+----------+-------+
|order_id|        company_name|  country|order_date|freight|
+--------+--------------------+---------+----------+-------+
|   10400|  Eastern Connection|       UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|      USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|  Austria|1997-01-02|  67.88|
|   10403|        Ernst Handel|  Austria|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|    Italy|1997-01-03| 155.97|
|   10405|    LINO-Delicateses|Venezuela|1997-01-06|  34.82|
|   10406|       Queen Cozinha|   Brazil|1997-01-07| 108.04|
|   10407|  Ottilies Käseladen|  Germany|1997-01-07|  91.48|
|   10408|   Folies gourmandes|   France|1997-01-08|  11.26|
|   10409|Océano Atlántico ...|Argentina|1997-01-09|  29.83|
+--------+--------------------+---------+----------+-------+
only showing top 10 rows

root
 |-- order_id: integer (nullable = true)
 |-- company_name: string (nullable = true)
 |-- country: string (nullable

##### A22 — Jointure order_details + products

In [126]:
# volumes pour silver 
DATA_PATH_silver = "/home/jovyan/data/tmp/silver" 

df_order_details_products = (
    df_order_details.alias("od")
    .join(df_products.alias("p"), on="product_id", how="inner")
    .select(
        "od.order_id",
        "od.product_id",
        "od.prix_unitaire",   # ancien unit_price
        "od.quantite",        # ancien quantity
        "od.discount",
        "p.product_name",
        "p.category_id",
        "p.unit_price"        # prix catalogue du produit
    )
)

#Vérification
#df_order_details_products.show(10)
df_order_details_products.printSchema()

# Écriture en Silver
df_order_details_products.write.mode("overwrite").parquet(f"{DATA_PATH_silver}/order_details_products")

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)



##### Jointure products + categories

In [127]:
# products_categories parquet file

DATA_PATH_silver = "/home/jovyan/data/tmp/silver" 

df_products_categories = (
    df_products.alias("p")
    .join(df_categories.alias("c"), on="category_id", how="left")
    .select(
        "p.product_id",
        "p.product_name",
        "p.supplier_id",
        "p.category_id",
        "c.category_name",
        "c.description",
        "p.quantity_per_unit",
        "p.unit_price",
        "p.units_in_stock",
        "p.units_on_order",
        "p.reorder_level",
        "p.discontinued"
    )
)
#Vérification

#df_products_categories.show(10)
df_products_categories.printSchema()

#Écriture en Silver
df_products_categories.write.mode("overwrite").parquet(f"{DATA_PATH_silver}/products_categories")


root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity_per_unit: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_in_stock: integer (nullable = true)
 |-- units_on_order: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- discontinued: integer (nullable = true)



##### DataFrame enrichi complet

In [128]:
# products enriched

DATA_PATH_silver = "/home/jovyan/data/tmp/silver"
#Enrichir les produits avec les catégories
df_products_enriched = (df_products.alias("p").join(df_categories.alias("c"), on="category_id", how="left"))
#Ecrire en silver
df_products_enriched.write.mode("overwrite").parquet(f"{DATA_PATH_silver}/products_enriched")

##### Rénomer les colonnes

In [143]:
# Création de volume bronze

# Définition du chemin Bronze
DATA_PATH_BRONZE = "/home/jovyan/data/tmp/bronze"

# Renaming of columns to avoid duplication

#customers 
df_customers = df_customers.withColumnRenamed("customer_id", "customer_id_customer")

#Employees
df_employees = (
    df_employees
    .withColumnRenamed("city", "employee_city")
    .withColumnRenamed("country", "employee_country")
    .withColumnRenamed("region", "employee_region")
    .withColumnRenamed("postal_code", "employee_postal_code")
)

# shippers
df_shippers = (
    df_shippers
    .withColumnRenamed("city", "shipper_city")
    .withColumnRenamed("phone", "shipper_phone")
    .withColumnRenamed("company_name", "shipper_name")
)

# Orders
df_orders = df_orders.withColumnRenamed("customer_id", "customer_id_order")

# products enriched

df_products_enriched = df_products_enriched.withColumnRenamed("customer_id", "customer_id_product")

# order_details

df_order_details = df_order_details.withColumnRenamed("customer_id", "customer_id_detail")

# products enriched
df_products_enriched = df_products_enriched.withColumnRenamed("customer_id", "customer_id_product")


# Jointure complète (order_details → orders → customers → products → employees → shippers)
# Jointure complète
df_full = (
    df_order_details.alias("od")
    .join(df_orders.alias("o"), od.customer_id_detail == o.customer_id_order, "inner")
    .join(df_customers.alias("cu"), o.customer_id_order == cu.customer_id_customer, "left")
    .join(df_products_enriched.alias("pr"), on="product_id", how="left")
    .join(df_employees.alias("e"), on="employee_id", how="left")
    .join(df_shippers.alias("s"), on="ship_via", how="left")
)

# Colonnes dupliquées
cols = df_full.columns
duplicates = [col for col, count in Counter(cols).items() if count > 1]

print("Colonnes dupliquées :", duplicates)

# Vérification
df_full.printSchema()

# Écriture en Bronze
df_full.write.mode("overwrite").parquet(f"{DATA_PATH_BRONZE}/full_join")


NameError: name 'od' is not defined

In [138]:

print(df_order_details.columns)



['order_id', 'product_id', 'prix_unitaire', 'quantite', 'discount', 'sous_total']
